In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ldf=pd.read_csv("loan.csv")
def understanding_columns(ldf):
    str_cols=ldf.select_dtypes(include=['object', 'string']).columns
    first_values = ldf[str_cols].iloc[0]
    result = pd.DataFrame({'column': str_cols, 'first_value': first_values.values})
    return result
#as we can see there are many columns with Object datatype and there may coem a need to convert them to float type if any groupby or aggregation needs to be done ahead
def convert_to_float(colname):
    return colname.apply(lambda x:0.0 if x == 0 else float(x))
    

def data_cleaning(ldf):
    cols_to_drop = ldf.columns[ldf.isnull().sum() == 39717].tolist()
    ldf=ldf.drop(columns=cols_to_drop)
    #further checking if any of the remaining columns have null values >50% of their rows as they will not offer much in analysis 
    cols_to_drop2= ldf.columns[(ldf.isnull().mean() *100)>50].tolist()
    ldf=ldf.drop(columns=cols_to_drop2)

    #fitering all string and object columns to perform basic cleaning like removing any whitespace, replacing wild characters, filling null with a specific string
    str_cols = ldf.select_dtypes(include=['object', 'string']).columns
    ldf[str_cols] = (
        ldf[str_cols]
        .apply(lambda col: col.str.strip())
        .apply(lambda col: col.str.lower())
        .replace(r'[%+]', '', regex=True)
        .fillna('unknown')
    )
    #clean numeric columns, filling them with median
    num_cols = ldf.select_dtypes(include=['number']).columns
    for col in num_cols:
        median_val = ldf[col].median()
        ldf[col] = ldf[col].fillna(median_val)

    ldf['loan_amnt'] =convert_to_float(ldf['loan_amnt'])
    ldf['int_rate']=convert_to_float(ldf['int_rate'])
    
    ldf['term']=ldf['term'].str.replace("months","")
    ldf['term']=convert_to_float(ldf['term'])
    
    #extract years from date columns
    ldf['issue_month']=pd.to_datetime(ldf['issue_d'],format='%b-%y',errors='coerce').dt.month

    #finally print the total null values in the dataframe, if its less than 10% then we are good to do EDA
    print(ldf.isnull().mean().sort_values(ascending=False) *100)

    return ldf

#method to save all the plots that we will generate using univariate, bivariate and multivariate analysis
#def save_plot(filename):
#    plt.tight_layout()
#    plt.savefig(f"plots/{filename}",dpi=300)
#    plt.close()


def univariate_analysis(ldf):
    #----Starting with Numeric variables -------------------
    #1. #Most important factor to identify risks is by analysing loan status. 
    # Below picture shows 14% are of loans are charged off ie. defaulted

    sns.countplot(x='loan_status',data=ldf)
    plt.title("Current loan Status")
    plt.show()

    counts= ldf['loan_status'].value_counts(normalize=True)*100
    print(counts)

    #2. Loan amount is also a very important varaible for a loan company to consider its overall risk
    # In our case study, we can clearly see most loans are taken between 1000-35000 and the concentration is specifically in small loans that is <20000 
    sns.displot(ldf['loan_amnt'])
    plt.title('Distribution of loan amount')
    plt.show()

    #3. Looking at the Interest rate of this company will tell its risk zones
    #the below plot shows us that 50% of loans fall between 10-15interest rate,hence its the most typical interest rate given
    sns.boxplot(y=ldf['int_rate'])
    plt.title('Distribution of interest rate')
    plt.show()

    #4. DTI =0 means no debt, 10= debr is 10% of the income, as the plot indicates, dti goes down at the end meaning very less borrowers are risky, which is common as lenders dont approve loans for people with high dti
    sns.histplot(ldf['dti'],kde=True,bins=30)
    plt.title('Distribution of Debt to income ratio')
    plt.xlabel('dti values')
    plt.ylabel('count of borrowers')
    plt.show()

    #Below categorical plots explain how loans are structured within this loan company
    #5. Grade
    sns.countplot(ldf['grade'])
    plt.title("Distribution of Loan Grades")
    plt.show()
    
    #6. understanding the reason for loan
    sns.countplot(y='purpose',data=ldf, order=ldf['purpose'])
    plt.title("Purpose of loan")
    plt.show()

    #7. Term
    sns.countplot(x="term",data=ldf)
    plt.title("Distribution of loan term")
    plt.show()


def bivariate_analysis(ldf):
    #1. grade vs loan status
    sns.countplot(data=ldf, x="grade",hue="loan_status", order=sorted(ldf['grade'].unique()))
    plt.title("Loan Status relationship with grade")
    plt.show()

    #2. Loan amount vs Loan status
    sns.boxplot(x="loan_status", y="loan_amnt",data=ldf)
    plt.title("Loan Status vs Amount")
    plt.show()

def multivariate_analysis(ldf):
    #to analyze the trend on loan default, we must first convert loan status in binary
    ldf['loan_status'].unique()
    ldf['default_flag']=ldf['loan_status'].isin(['charged off','default']).astype(int)  

    default_rate=ldf.groupby('grade')['default_flag'].mean().reset_index()
    # this plot proves grade g defaults the highest and are most risky
    sns.barplot(x='grade',y='default_flag',data=default_rate)
    plt.title("Grade Defaulters")
    plt.show()
    #check if pricing matches the risk above, grade vs int rate;#below graph indicates the strategy of loan department is appropriate as the interest rate is increasing from better-worse grades

    sns.barplot(x='grade', y='int_rate',data=ldf,order=sorted(ldf['grade']))
    plt.title("Grade vs interestRate")
    plt.show()

    #correlation 
    corr=ldf[['loan_amnt','dti','int_rate','annual_inc']].corr()
    sns.heatmap(corr,annot=True,cmap='coolwarm')
    plt.title("Correlation between important numeric vars")
    plt.show()

if __name__ == "__main__":
    print("Starting Loan EDA...")
    understanding_columns(ldf)
    ldf=data_cleaning(ldf)
    univariate_analysis(ldf)
    bivariate_analysis(ldf)
    multivariate_analysis(ldf)